[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH06/06_01_begin/notebooks/06_01_compare_to_baseline.ipynb)

# 06_01 Compare your model to the baseline

Run the base model and your fine-tuned model on every evaluation ticket, then compare them on triage accuracy, format, and voice.

Before you start, choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%%capture
# Install Unsloth. This takes a minute or two.
!pip install unsloth

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH06")

import ridgeline

## Choose your adapter

In [ ]:
# Set this to False if you didn't save your own adapter in Chapter 5.
USE_MY_ADAPTER = True

MY_ADAPTER = "/content/drive/MyDrive/ridgeline-lora"
INSTRUCTOR_ADAPTER = "KWillz/ridgeline-support-lora"

if USE_MY_ADAPTER:
    from google.colab import drive
    drive.mount("/content/drive")
    adapter = MY_ADAPTER
else:
    adapter = INSTRUCTOR_ADAPTER
print(f"Using adapter: {adapter}")

## Load the fine-tuned model

In [ ]:
from unsloth import FastLanguageModel

# Loading an adapter also loads the base model it was trained on.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.for_inference(model)

## Load the evaluation tickets

In [ ]:
eval_tickets = ridgeline.load_eval_tickets()
print(f"{len(eval_tickets)} evaluation tickets. The model never trained on any of them.")

## Run both models on every ticket

In [ ]:
# TODO: For each ticket, generate a baseline reply with the adapter disabled and the voice rules in the prompt, then a fine-tuned reply with no voice rules. Store both in outputs.


## Score both models

In [ ]:
# TODO: Score both models with ridgeline.check_reply and show category, escalate, header format, and voice pass rates in a table.


## Read the category misses

A percentage hides the details. Print every ticket where the fine-tuned model chose a different category than the label, and read the subjects. Some misses are real mistakes. Others are tickets where the label itself is debatable.

In [ ]:
for t, o in zip(eval_tickets, outputs):
    got = ridgeline.parse_output(o["finetuned"])["category"]
    if got != t["category"]:
        print(f"{t['id']:9} expected {t['category']:20} got {got}  | {t['subject'][:50]}")

## Compare prompt sizes

In [ ]:
prompt_sizes = {
    "Baseline": ridgeline.count_tokens(tokenizer, ridgeline.build_messages(eval_tickets[0], True)),
    "Fine-tuned": ridgeline.count_tokens(tokenizer, ridgeline.build_messages(eval_tickets[0], False)),
}
for name, size in prompt_sizes.items():
    print(f"{name} prompt: {size} tokens per ticket")

## Look at one ticket side by side

In [ ]:
ticket = next(t for t in eval_tickets if t["id"] == "RT-008")
pair = next(o for o in outputs if o["id"] == "RT-008")
print(f"{ticket['subject']}\n")
print("BASELINE\n" + pair["baseline"] + "\n")
print("FINE-TUNED\n" + pair["finetuned"])

## Save the outputs

In [ ]:
ridgeline.save_jsonl(outputs, "eval_outputs.jsonl")
if USE_MY_ADAPTER:
    ridgeline.save_jsonl(outputs, "/content/drive/MyDrive/eval_outputs.jsonl")
print("Saved eval_outputs.jsonl. You'll score it in the challenge.")